# Feature Engineering
In this notebook, we construct datetime, holiday, lag, and rolling features, ensuring strict avoidance of data leakage.

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import holidays

print("Imports ready.")

Imports ready.


In [2]:
df = pd.read_csv('../data/raw/train.csv', parse_dates=['date'])
df = df.sort_values(['store', 'item', 'date']).reset_index(drop=True)

print(f"Loaded: {df.shape}")
df.head()

Loaded: (913000, 4)


,date,store,item,sales
0,2013-01-01,1,1,2
1,2013-01-02,1,1,2
2,2013-01-03,1,1,3
3,2013-01-04,1,1,5
4,2013-01-05,1,1,4


In [3]:
from src.features import add_datetime_features

df = add_datetime_features(df)
print(f"Shape after datetime features: {df.shape}")
print("New columns:", [c for c in df.columns if c not in ['date', 'store', 'item', 'sales']])

Shape after datetime features: (913000, 21)
New columns: ['year', 'month', 'day', 'dayofweek', 'dayofyear', 'weekofyear', 'quarter', 'is_weekend', 'is_month_start', 'is_month_end', 'is_quarter_end', 'month_sin', 'month_cos', 'dayofweek_sin', 'dayofweek_cos', 'dayofyear_sin', 'dayofyear_cos']


In [4]:
from src.features import add_holiday_features

df = add_holiday_features(df, country='US')
print("Holiday features added.")

Holiday features added.


In [5]:
from src.features import add_lag_features

df = add_lag_features(df, lags=[1, 7, 14, 21, 28, 91, 365])
print("Lag features added.")

Lag features added.


In [6]:
from src.features import add_rolling_features

df = add_rolling_features(df, windows=[7, 14, 30, 90])
print("Rolling features added.")

Rolling features added.


In [7]:
from src.features import add_expanding_features

df = add_expanding_features(df)
print("Expanding features added.")

Expanding features added.


In [8]:
# Drop rows with NaN
initial_rows = len(df)
df_features = df.dropna().reset_index(drop=True)
dropped = initial_rows - len(df_features)

print(f"Dropped {dropped:,} rows due to NaN in lag/rolling features")
print(f"Remaining rows: {len(df_features):,}")


Dropped 182,500 rows due to NaN in lag/rolling features
Remaining rows: 730,500


In [9]:
df_features.to_csv('../data/processed/features.csv', index=False)
print("Saved processed features to ../data/processed/features.csv")

Saved processed features to ../data/processed/features.csv
